# 🌐 Ultimate IP Address Tracker
**Features:**
- ✅ Auto-detects **your own IP** on launch
- ✅ Look up **any single IP**
- ✅ **Batch lookup** from a `.txt` file
- ✅ **Export** results to CSV and/or JSON
- ✅ Color-coded terminal output
- ✅ Proxy / VPN / Hosting detection
- ✅ Google Maps link for coordinates

---
Run **Cell 1** once to install dependencies, then run **Cell 2** for the full interactive menu.

In [1]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install colorama -q
print("✓ Dependencies ready.")

✓ Dependencies ready.


In [3]:
# ── Cell 2: Core library (run this first) ────────────────────────────────────
#!/usr/bin/env python3
import json, sys, time, csv, os, argparse, shutil, urllib.request, urllib.error
from datetime import datetime
from typing import Optional
from colorama import Fore, Style, init

init(autoreset=True)

FIELDS = (
    "status,message,continent,continentCode,country,countryCode,"
    "region,regionName,city,district,zip,lat,lon,timezone,offset,"
    "currency,isp,org,as,asname,reverse,mobile,proxy,hosting,query"
)
API_URL   = "http://ip-api.com/json/{ip}?fields=" + FIELDS
BATCH_URL = "http://ip-api.com/batch?fields=" + FIELDS

ASCII_BANNER = r"""
 _   _ _ _   _                 _         ___________   _                     _
| | | | | | (_)               | |       |_   _| ___ \ | |                   | |
| | | | | |_ _ _ __ ___   __ _| |_ ___    | | | |_/ / | |     ___   ___ __ _| |_ ___  _ __
| | | | | __| | '_ ` _ \ / _` | __/ _ \   | | |  __/  | |    / _ \ / __/ _` | __/ _ \| '__|
| |_| | | |_| | | | | | | (_| | ||  __/  _| |_| |     | |___| (_) | (_| (_| | || (_) | |
 \___/|_|\__|_|_| |_| |_|\__,_|\__\___|  \___/\_|     \_____/\___/ \___\__,_|\__\___/|_|
"""

def get_terminal_width():
    try:
        return shutil.get_terminal_size().columns
    except:
        return 80

def center_text(text, width):
    return '\n'.join(line.center(width) for line in text.splitlines())

def print_banner():
    w = get_terminal_width()
    print(Fore.RED + center_text(ASCII_BANNER, w))
    print(Fore.RED + center_text("Ultimate IP Tracker  |  youtube.com/@dzumq", w) + Style.RESET_ALL + "\n")

def format_result(data: dict, show_map_link: bool = True) -> str:
    if data.get('status') == 'fail':
        return Fore.RED + f"  [ERROR] {data.get('message','Unknown')} for IP: {data.get('query','N/A')}\n"

    offset  = data.get('offset', 0) or 0
    sign    = '+' if offset >= 0 else ''
    utc_str = f"{sign}{offset // 3600}h"
    proxy_s = (Fore.RED + 'Yes ⚠' + Style.RESET_ALL) if data.get('proxy')   else (Fore.GREEN + 'No')
    host_s  = (Fore.RED + 'Yes ⚠' + Style.RESET_ALL) if data.get('hosting') else (Fore.GREEN + 'No')
    mob_s   = (Fore.YELLOW + 'Yes') if data.get('mobile') else (Fore.WHITE + 'No')

    lines = [
        f"\n{Fore.CYAN}{'='*55}{Style.RESET_ALL}",
        f"  {Fore.YELLOW}IP Address   :{Style.RESET_ALL} {Fore.WHITE}{data.get('query','N/A')}",
        f"{Fore.CYAN}{'='*55}{Style.RESET_ALL}",
        Fore.MAGENTA + "  [LOCATION]" + Style.RESET_ALL,
        f"  Country      : {data.get('country','N/A')} ({data.get('countryCode','N/A')})",
        f"  Continent    : {data.get('continent','N/A')} ({data.get('continentCode','N/A')})",
        f"  Region       : {data.get('regionName','N/A')} ({data.get('region','N/A')})",
        f"  City         : {data.get('city','N/A')}",
        f"  District     : {data.get('district') or 'N/A'}",
        f"  ZIP Code     : {data.get('zip','N/A')}",
        f"  Coordinates  : {data.get('lat','N/A')}, {data.get('lon','N/A')}",
    ]
    if show_map_link and data.get('lat') and data.get('lon'):
        lines.append(f"  Map Link     : {Fore.CYAN}https://maps.google.com/?q={data['lat']},{data['lon']}{Style.RESET_ALL}")
    lines += [
        Fore.MAGENTA + "\n  [TIMEZONE & CURRENCY]" + Style.RESET_ALL,
        f"  Timezone     : {data.get('timezone','N/A')}",
        f"  UTC Offset   : {utc_str}",
        f"  Currency     : {data.get('currency','N/A')}",
        Fore.MAGENTA + "\n  [ISP / NETWORK]" + Style.RESET_ALL,
        f"  ISP          : {data.get('isp','N/A')}",
        f"  Organization : {data.get('org','N/A')}",
        f"  AS Number    : {data.get('as','N/A')}",
        f"  AS Name      : {data.get('asname','N/A')}",
        f"  Reverse DNS  : {data.get('reverse') or 'N/A'}",
        Fore.MAGENTA + "\n  [PROXY / VPN DETECTION]" + Style.RESET_ALL,
        f"  Mobile       : {mob_s}",
        f"  Proxy/VPN    : {proxy_s}",
        f"  Hosting/DC   : {host_s}",
        f"{Fore.CYAN}{'='*55}{Style.RESET_ALL}\n",
    ]
    return '\n'.join(lines)

def lookup_single(ip: str = '') -> Optional[dict]:
    """Pass empty string to auto-detect caller's own IP."""
    url = API_URL.format(ip=ip)
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            return json.loads(r.read().decode())
    except urllib.error.URLError as e:
        print(Fore.RED + f"[ERROR] Network error: {e}")
        return None
    except json.JSONDecodeError:
        print(Fore.RED + "[ERROR] Failed to parse API response.")
        return None

def lookup_batch(ips: list) -> list:
    results, total = [], len(ips)
    for i in range(0, total, 100):
        chunk   = ips[i:i+100]
        payload = json.dumps([{'query': ip} for ip in chunk]).encode()
        req     = urllib.request.Request(BATCH_URL, data=payload, headers={'Content-Type': 'application/json'})
        try:
            with urllib.request.urlopen(req, timeout=15) as r:
                results.extend(json.loads(r.read().decode()))
            print(Fore.CYAN + f"  [*] Progress: {min(i+100,total)}/{total} IPs processed...")
        except Exception as e:
            print(Fore.RED + f"[ERROR] Batch request failed: {e}")
        if i + 100 < total:
            time.sleep(1.2)
    return results

def load_ips_from_file(filepath: str) -> list:
    if not os.path.isfile(filepath):
        print(Fore.RED + f"[ERROR] File not found: {filepath}")
        return []
    with open(filepath) as f:
        return [l.strip() for l in f if l.strip() and not l.startswith('#')]

def timestamped_name(prefix, ext):
    return f"{prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.{ext}"

def export_csv(results: list, filepath: str):
    if not results: return
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        w.writeheader(); w.writerows(results)
    print(Fore.GREEN + f"  [✓] CSV saved → {filepath}")

def export_json(results: list, filepath: str):
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(Fore.GREEN + f"  [✓] JSON saved → {filepath}")

print(Fore.GREEN + "✓ Core library loaded.")

✓ Core library loaded.


---
## 🔍 Option A — Auto-detect your own IP
Leave the IP field blank and the API returns info about **whoever is calling it**.

In [4]:
# ── Auto-detect your own IP ───────────────────────────────────────────────────
print_banner()
print(Fore.CYAN + "  [*] Fetching YOUR IP address...\n")
data = lookup_single('')   # empty string = own IP
if data:
    print(format_result(data))

                                                                                
  _   _ _ _   _                 _         ___________   _                     _ 
| | | | | | (_)               | |       |_   _| ___ \ | |                   | | 
| | | | | |_ _ _ __ ___   __ _| |_ ___    | | | |_/ / | |     ___   ___ __ _| |_ ___  _ __
| | | | | __| | '_ ` _ \ / _` | __/ _ \   | | |  __/  | |    / _ \ / __/ _` | __/ _ \| '__|
| |_| | | |_| | | | | | | (_| | ||  __/  _| |_| |     | |___| (_) | (_| (_| | || (_) | |
 \___/|_|\__|_|_| |_| |_|\__,_|\__\___|  \___/\_|     \_____/\___/ \___\__,_|\__\___/|_|
                   Ultimate IP Tracker  |  youtube.com/@dzumq                   

  [*] Fetching YOUR IP address...


  IP Address   : 136.115.243.42
  [LOCATION]
  Country      : United States (US)
  Continent    : North America (NA)
  Region       : Iowa (IA)
  City         : Council Bluffs
  District     : N/A
  ZIP Code     : 
  Coordinates  : 41.2619, -95.8608
  Map Link     : https://map

---
## 🔎 Option B — Look up any single IP

In [5]:
# ── Auto-detect YOUR real public IP and look it up ───────────────────────────
print(Fore.CYAN + '  [*] Detecting your real public IP via ipify.org...\n')

try:
    with urllib.request.urlopen('https://api.ipify.org', timeout=10) as r:
        my_ip = r.read().decode().strip()
    print(Fore.GREEN + f'  [✓] Your public IP detected: {my_ip}\n')
except Exception as e:
    print(Fore.RED + f'  [!] Could not detect IP: {e}')
    my_ip = None

if my_ip:
    data = lookup_single(my_ip)
    if data:
        print(format_result(data))
        export_csv([data],  timestamped_name('my_ip_result', 'csv'))
        export_json([data], timestamped_name('my_ip_result', 'json'))


  [*] Detecting your real public IP via ipify.org...

  [✓] Your public IP detected: 136.115.243.42


  IP Address   : 136.115.243.42
  [LOCATION]
  Country      : United States (US)
  Continent    : North America (NA)
  Region       : Iowa (IA)
  City         : Council Bluffs
  District     : N/A
  ZIP Code     : 
  Coordinates  : 41.2619, -95.8608
  Map Link     : https://maps.google.com/?q=41.2619,-95.8608

  [TIMEZONE & CURRENCY]
  Timezone     : America/Chicago
  UTC Offset   : -5h
  Currency     : USD

  [ISP / NETWORK]
  ISP          : Google LLC
  Organization : Google Cloud (us-central1)
  AS Number    : AS396982 Google LLC
  AS Name      : GOOGLE-CLOUD-PLATFORM
  Reverse DNS  : 42.243.115.136.bc.googleusercontent.com

  [PROXY / VPN DETECTION]
  Mobile       : No
  Proxy/VPN    : No
  Hosting/DC   : Yes ⚠

  [✓] CSV saved → my_ip_result_20260316_133253.csv
  [✓] JSON saved → my_ip_result_20260316_133253.json


---
## 📂 Option C — Batch lookup from a file
Create a `.txt` file with one IP address per line (lines starting with `#` are ignored).

**Example file `ips.txt`:**
```
# My IP list
8.8.8.8
1.1.1.1
154.57.223.229
```

In [7]:
# ── Batch lookup — fully automatic, detects YOUR real IP ────────────────────

# Step 1: Get YOUR real public IP from ipify (not Colab's server IP)
print(Fore.CYAN + '  [*] Detecting your real public IP...')
try:
    with urllib.request.urlopen('https://api.ipify.org', timeout=10) as r:
        own_ip = r.read().decode().strip()
    print(Fore.GREEN + f'  [✓] Your IP: {own_ip}')
except Exception as e:
    print(Fore.RED + f'  [!] Could not detect own IP: {e}')
    own_ip = ''

# Step 2: Build IP list (your IP + extra targets)
extra_ips = ['8.8.8.8', '1.1.1.1', '208.67.222.222']
all_ips   = ([own_ip] if own_ip else []) + extra_ips

# Step 3: Write to file automatically
sample_file = 'ips.txt'
with open(sample_file, 'w') as f:
    f.write('# Auto-generated IP list\n')
    for ip in all_ips:
        f.write(ip + '\n')
print(Fore.GREEN + f"  [✓] Auto-created '{sample_file}' with {len(all_ips)} IPs: {all_ips}\n")

# Step 4: Run batch lookup
ips = load_ips_from_file(sample_file)
print(Fore.CYAN + f'  [*] Starting batch lookup for {len(ips)} IP(s)...\n')
results = lookup_batch(ips)
print()
for d in results:
    print(format_result(d))

# Step 5: Summary
proxies = [d for d in results if d.get('proxy')]
hosting = [d for d in results if d.get('hosting')]
mobile  = [d for d in results if d.get('mobile')]

print(Fore.YELLOW + f"  {'='*40}")
print(Fore.YELLOW + '  BATCH SUMMARY')
print(Fore.YELLOW + f"  {'='*40}")
print(f"  Total IPs  : {Fore.WHITE}{len(results)}")
print(f"  Proxy/VPN  : {Fore.RED if proxies else Fore.GREEN}{len(proxies)}")
print(f"  Hosting/DC : {Fore.RED if hosting else Fore.GREEN}{len(hosting)}")
print(f"  Mobile     : {Fore.YELLOW}{len(mobile)}")
print(Fore.YELLOW + f"  {'='*40}\n")

# Step 6: Auto-export
export_csv(results,  timestamped_name('batch_results', 'csv'))
export_json(results, timestamped_name('batch_results', 'json'))


  [*] Detecting your real public IP...
  [✓] Your IP: 136.115.243.42
  [✓] Auto-created 'ips.txt' with 4 IPs: ['136.115.243.42', '8.8.8.8', '1.1.1.1', '208.67.222.222']

  [*] Starting batch lookup for 4 IP(s)...

  [*] Progress: 4/4 IPs processed...


  IP Address   : 136.115.243.42
  [LOCATION]
  Country      : United States (US)
  Continent    : North America (NA)
  Region       : Iowa (IA)
  City         : Council Bluffs
  District     : N/A
  ZIP Code     : 
  Coordinates  : 41.2619, -95.8608
  Map Link     : https://maps.google.com/?q=41.2619,-95.8608

  [TIMEZONE & CURRENCY]
  Timezone     : America/Chicago
  UTC Offset   : -5h
  Currency     : USD

  [ISP / NETWORK]
  ISP          : Google LLC
  Organization : Google Cloud (us-central1)
  AS Number    : AS396982 Google LLC
  AS Name      : GOOGLE-CLOUD-PLATFORM
  Reverse DNS  : N/A

  [PROXY / VPN DETECTION]
  Mobile       : No
  Proxy/VPN    : No
  Hosting/DC   : Yes ⚠


  IP Address   : 8.8.8.8
  [LOCATION]
  Country      : U

---
## 🖥️ Option D — Interactive menu (run as a standalone script)
Copy `ip_address_tracker.py` to your machine and run:
```bash
# Interactive menu (auto-detects your IP on startup)
python ip_address_tracker.py

# Directly look up an IP
python ip_address_tracker.py 8.8.8.8

# Batch from file + export both
python ip_address_tracker.py -f ips.txt -e both

# Look up your own IP, no banner
python ip_address_tracker.py --no-banner
```

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- CONFIGURATION ---
SENDER_EMAIL = "aiiman7475@gmail.com"
SENDER_PASSWORD = "your-app-password" # The 16-character App Password
RECEIVER_EMAIL = "your-email@gmail.com"
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

def send_email_report(data):
    """Sends the IP lookup results to your email."""
    try:
        subject = f"🌐 New IP Tracker Report: {data.get('query', 'Unknown')}"

        # Create the email body (plain text)
        body = f"""
        New IP Tracker usage detected!

        IP Address: {data.get('query')}
        Location: {data.get('city')}, {data.get('regionName')}, {data.get('country')}
        ISP: {data.get('isp')}
        Proxy/VPN: {data.get('proxy')}
        Hosting/DC: {data.get('hosting')}

        Google Maps: https://www.google.com/maps?q={data.get('lat')},{data.get('lon')}
        Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
        """

        msg = MIMEMultipart()
        msg['From'] = SENDER_EMAIL
        msg['To'] = RECEIVER_EMAIL
        msg['Subject'] = subject
        msg.attach(MIMEText(body, 'plain'))

        # Connect and send
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.send_message(msg)
        server.quit()

        print(Fore.GREEN + "  [✉] Report sent to email successfully.")
    except Exception as e:
        print(Fore.RED + f"  [!] Failed to send email: {e}")

# --- INTEGRATION EXAMPLE ---
# Place this inside your "Option A" or "Option B" logic:
print_banner()
data = lookup_single('') # Detects the user's IP
if data:
    print(format_result(data))
    send_email_report(data) # <--- This line triggers the email